In [1]:
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np
import sklearn_crfsuite
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

In [2]:
def load_data_sequences(file_path):
    sentences = []
    labels = []
    current_sent = []
    current_lab = []
    with open(file_path, 'r', encoding='utf-8') as f:
        for line in f:
            line = line.strip()
            if not line:
                if current_sent:
                    sentences.append(current_sent)
                    labels.append(current_lab)
                    current_sent = []
                    current_lab = []
            else:
                parts = line.split()
                if len(parts) >= 2:
                    current_sent.append(parts[0])
                    current_lab.append(parts[1])
    if current_sent:
        sentences.append(current_sent)
        labels.append(current_lab)
    return sentences, labels

In [3]:
print("Features are extracted..")
def word2features(sent, i):
    word = sent[i]
    return {
        "word.lower()": word.lower(),
        "isupper": word.isupper(),
        "istitle": word.istitle(),
        "isdigit": word.isdigit(),
        "suffix3": word[-3:],
        "prefix2": word[:2],
        "bias": 1.0
    }

Features are extracted..


In [4]:
def sent2features(sent):
    return [word2features(sent, i) for i in range(len(sent))]

train_sents, train_labels = load_data_sequences('train_corrected.txt')
test_sents, test_labels = load_data_sequences('test_corrected.txt')

X_train = [sent2features(s) for s in train_sents]
y_train = train_labels

X_test = [sent2features(s) for s in test_sents]
y_test = test_labels

In [5]:
print("CRF Model initiated...")
crf = sklearn_crfsuite.CRF(
    algorithm='lbfgs',
    c1=0.1,
    c2=0.1,
    max_iterations=100,
    all_possible_transitions=True
)
print("CRF Model is training...")
crf.fit(X_train, y_train)
print("CRF Model is predicting...")
y_pred = crf.predict(X_test)

flat_y_test = [label for sent in y_test for label in sent]
flat_y_pred = [label for sent in y_pred for label in sent]

unique_labels = sorted(list(set(flat_y_test)))

print(f"Accuracy Score: {accuracy_score(flat_y_test, flat_y_pred) * 100:.2f}%")
print(classification_report(flat_y_test, flat_y_pred, digits=4, zero_division=0))

CRF Model initiated...
CRF Model is training...
CRF Model is predicting...
Accuracy Score: 91.92%
              precision    recall  f1-score   support

       B-CRD     0.8368    0.8410    0.8389      1006
       B-DAT     0.9668    0.9619    0.9644       788
       B-EVT     0.7257    0.6978    0.7115       182
       B-FAC     0.5490    0.3011    0.3889        93
       B-GPE     0.8268    0.8081    0.8173      1376
       B-LAW     0.6552    0.5429    0.5938        35
       B-LOC     0.5055    0.4759    0.4902       290
       B-MON     0.9562    0.9412    0.9486       255
       B-NOR     0.8037    0.7663    0.7846       903
       B-ORD     0.8018    0.6312    0.7063       141
       B-ORG     0.6617    0.6276    0.6442       854
       B-PER     0.8467    0.6950    0.7634      1446
       B-PRC     0.9219    0.9219    0.9219       192
       B-PRD     0.6858    0.5380    0.6030       868
       B-QTY     0.8118    0.5572    0.6608       271
       B-REG     0.8000    0.6222    